In [59]:
import re

import catboost
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import plotly.express as px
import seaborn as sns
import shap
from catboost import CatBoostClassifier, Pool
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


In [2]:
master_df = pd.read_csv('../data/csv/master_df.csv')
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 220688 entries, 0 to 220687
Data columns (total 57 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id                        220688 non-null  str    
 1   href                      220688 non-null  str    
 2   price                     220688 non-null  int64  
 3   title                     220688 non-null  str    
 4   status                    220688 non-null  str    
 5   content                   220688 non-null  str    
 6   createdAt                 220688 non-null  str    
 7   boostedAt                 220688 non-null  str    
 8   user_dbId                 220688 non-null  int64  
 9   user_nickname             220688 non-null  str    
 10  region_name_from_article  220688 non-null  str    
 11  region_id                 220688 non-null  int64  
 12  region_name               220688 non-null  str    
 13  region_in                 220688 non-null  str    
 14 

In [3]:
cols = [
    'id', 'price', 'title', 'status', 'content', 'createdAt',
    'boostedAt', 'region_name', 'crawledAt', 'favoriteCount',
    'chatCount', 'viewCount', 'sellerTemperature', 'imageCount',
    'brandName', 'label', 'coarse_label', 'days_since_created',
    'created_hour', 'created_dayofweek', 'is_boosted', 'title_len',
    'content_len', 'days_elapsed'
]

In [5]:
### feature engineering ###
df = master_df[cols]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 220688 entries, 0 to 220687
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  220688 non-null  str    
 1   price               220688 non-null  int64  
 2   title               220688 non-null  str    
 3   status              220688 non-null  str    
 4   content             220688 non-null  str    
 5   createdAt           220688 non-null  str    
 6   boostedAt           220688 non-null  str    
 7   region_name         220688 non-null  str    
 8   crawledAt           220688 non-null  str    
 9   favoriteCount       220688 non-null  int64  
 10  chatCount           220688 non-null  int64  
 11  viewCount           220688 non-null  int64  
 12  sellerTemperature   220688 non-null  float64
 13  imageCount          220688 non-null  int64  
 14  brandName           220688 non-null  str    
 15  label               220688 non-null  str    


In [7]:
df.to_csv('../data/csv/parse_brands.csv')

In [9]:
df = pd.read_csv('../data/csv/parse_brands_enriched.csv', index_col=0)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 220688 entries, 0 to 220687
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  220688 non-null  str    
 1   price               220688 non-null  int64  
 2   title               220688 non-null  str    
 3   status              220688 non-null  str    
 4   content             220688 non-null  str    
 5   createdAt           220688 non-null  str    
 6   boostedAt           220688 non-null  str    
 7   region_name         220688 non-null  str    
 8   crawledAt           220688 non-null  str    
 9   favoriteCount       220688 non-null  int64  
 10  chatCount           220688 non-null  int64  
 11  viewCount           220688 non-null  int64  
 12  sellerTemperature   220688 non-null  float64
 13  imageCount          220688 non-null  int64  
 14  brandName           220688 non-null  str    
 15  label               220688 non-null  str    


In [38]:
# 7일 이내 올라온 게시글 중에 가격이 1이고 판매된 데이터 필터링
filtered_df = df[(df['price'] == 1) & (df['status'] != 'Ongoing') & (df['days_elapsed'] <= 7)]

# 개수 확인
count = len(filtered_df)
print(count)

# 7일 이내 올라온 게시글 중에 가격이 1이고 판매된 데이터 필터링
filtered_df = df[(df['price'] == 1) & (df['status'] != 'Ongoing') & (df['days_elapsed'] > 7)]

# 개수 확인
count = len(filtered_df)
print(count)
## 1. 데이터 셋에서 판매중인 1원 상품 수는 33개, 판매 완료된 1원인 상품 수가 21 + 2223개이기 때문에
##    데이터중 노이즈라고 보고 빼는게 맞는것 같다.
## 2. 7일이 지난 판매중인 글 중에서도 33개 있기 때문에 완료돼서 바꾼 사람도 있고
##    나눔을 그냥 1원 받고 올린 사람도 있는것 같다.
## feedback. 향후에 1원인 데이터들을 제대로 카테고리별로 보고 뺄지 안뺄지 판단하면 될 듯 하다.

21
2223


In [39]:
# price가 1이 아닌 로우만 선택하여 저장
df = df[df['price'] != 1]

In [40]:
df['favorite_per_view'] = df['favoriteCount'] / (df['viewCount'] + 1)
df['chat_per_view'] = df['chatCount'] / (df['viewCount'] + 1)

In [41]:
brand_median_price = df.groupby('brandName')['price'].transform('median')
label_median_price = df.groupby('label')['price'].transform('median')

df['price_ratio_to_brand'] = df['price'] / (brand_median_price + 1)
df['price_ratio_to_label'] = df['price'] / (label_median_price + 1)

In [49]:
n_days = 7  # 예: 7일 이내 판매 여부

# 2. Target 변수 생성 함수
def make_target(row, n):
    # status 컬럼의 '거래완료'를 의미하는 정확한 텍스트 확인 필요 (예: 'Completed', 'Sold' 등)
    is_sold = row['status'] != 'Ongoing'  # 혹은 status_detail 사용

    if is_sold and row['days_elapsed'] <= n:
        return 1  # n일 이내 판매됨
    elif not is_sold and row['days_elapsed'] > n:
        return 0  # n일이 지났는데도 안 팔림
    else:
        return np.nan  # 아직 n일이 안 지났는데 안 팔린 상태 (보류)


df['target_n_days'] = df.apply(lambda x: make_target(x, n_days), axis=1)
# 판별 불가능한(보류된) 데이터 학습에서 제외
df = df.dropna(subset=['target_n_days']).copy()

In [50]:
df.info()

<class 'pandas.DataFrame'>
Index: 131788 entries, 0 to 220687
Data columns (total 29 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    131788 non-null  str    
 1   price                 131788 non-null  int64  
 2   title                 131788 non-null  str    
 3   status                131788 non-null  str    
 4   content               131788 non-null  str    
 5   createdAt             131788 non-null  str    
 6   boostedAt             131788 non-null  str    
 7   region_name           131788 non-null  str    
 8   crawledAt             131788 non-null  str    
 9   favoriteCount         131788 non-null  int64  
 10  chatCount             131788 non-null  int64  
 11  viewCount             131788 non-null  int64  
 12  sellerTemperature     131788 non-null  float64
 13  imageCount            131788 non-null  int64  
 14  brandName             131788 non-null  str    
 15  label           

In [51]:
cols_to_drop = [
    'id', 'status', 'createdAt', 'boostedAt', 'crawledAt', 'days_since_created', 
    'created_hour', 'created_dayofweek'
]

In [52]:
clean_df = df.drop(cols_to_drop, axis=1)
clean_df.info()

<class 'pandas.DataFrame'>
Index: 131788 entries, 0 to 220687
Data columns (total 21 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   price                 131788 non-null  int64  
 1   title                 131788 non-null  str    
 2   content               131788 non-null  str    
 3   region_name           131788 non-null  str    
 4   favoriteCount         131788 non-null  int64  
 5   chatCount             131788 non-null  int64  
 6   viewCount             131788 non-null  int64  
 7   sellerTemperature     131788 non-null  float64
 8   imageCount            131788 non-null  int64  
 9   brandName             131788 non-null  str    
 10  label                 131788 non-null  str    
 11  coarse_label          131788 non-null  str    
 12  is_boosted            131788 non-null  int64  
 13  title_len             131788 non-null  int64  
 14  content_len           131788 non-null  int64  
 15  days_elapsed    

In [56]:
X = clean_df.drop('target_n_days', axis=1)
y = clean_df['target_n_days']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_features = ['region_name', 'brandName', 'label', 'coarse_label']
text_features = ['title', 'content']

# ★ Optuna 속도 최적화를 위해 Pool 객체 미리 생성 ★
train_pool = Pool(
    X_train, y_train, cat_features=cat_features, text_features=text_features
)
valid_pool = Pool(
    X_test, y_test, cat_features=cat_features, text_features=text_features
)

In [ ]:
# ==========================================
# 2. Optuna Objective 함수 정의
# ==========================================

def objective(trial):
    # 탐색할 하이퍼파라미터 공간 정의
    param = {
        'iterations': trial.suggest_int('iterations', 500, 1500),  # 학습 횟수
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth': trial.suggest_int(
            'depth', 4, 8
        ),  # 텍스트 모델은 메모리를 많이 쓰므로 최대 8 권장
        'l2_leaf_reg': trial.suggest_float(
            'l2_leaf_reg', 1e-3, 10.0, log=True
        ),  # 정규화
        'random_strength': trial.suggest_float(
            'random_strength', 1e-3, 10.0, log=True
        ),  # 과적합 방지
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'bootstrap_type': 'Bayesian',  # 🌟 수정됨: GPU 환경에서 bagging_temperature 쓸 때 필수
        # 고정 파라미터
        'eval_metric': 'Logloss',
        'random_seed': 42,
        #'auto_class_weights': 'Balanced',
        'task_type': 'GPU',
        'verbose': 0,  # 튜닝 중에는 로그 출력 끄기
    }

    model = CatBoostClassifier(**param)

    # 모델 학습 (Early Stopping 적용)
    model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=50)

    # 검증셋에서 가장 높았던 PR-AUC 점수를 반환
    proba = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, proba)

    return pr_auc


# ==========================================
# 3. Optuna Study 실행 (탐색 시작)
# ==========================================
print('🚀 Optuna 하이퍼파라미터 탐색을 시작합니다...')
study = optuna.create_study(direction='maximize')  # AUC는 높을수록 좋으므로 maximize
study.optimize(
    objective, n_trials=30
)  # 우선 30번만 시도 (시간에 따라 50~100번으로 늘려보세요)

print('\n🏆 [Best Trial]')
print(f'최고 PR-AUC 점수: {study.best_value:.4f}')
print('최적의 파라미터:')
for key, value in study.best_params.items():
    print(f'  {key}: {value}')

# ==========================================
# 4. 찾은 최적 파라미터로 최종 모델 학습 및 평가
# ==========================================
print('\n🔥 찾은 최적의 파라미터로 최종 모델을 학습합니다...')
best_params = study.best_params

# 고정 파라미터 다시 추가
best_params.update(
    {
        'eval_metric': 'Logloss',
        'random_seed': 42,
        #'auto_class_weights': 'Balanced',
        'task_type': 'GPU',
        'verbose': 50,  # 최종 학습이므로 로그 출력
    }
)

final_model = CatBoostClassifier(**best_params)
final_model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=50)

# 최종 평가
preds = final_model.predict(X_test)
proba = final_model.predict_proba(X_test)[:, 1]

print('\n✅ [최종 모델 평가 지표]')
print(classification_report(y_test, preds))
print(f'PR-AUC: {average_precision_score(y_test, proba):.4f}')

# ==========================================
# 5. Permutation Importance 계산 (최종 1회)
# ==========================================
print('\n📊 Permutation Importance 계산 중... (시간이 조금 소요됩니다)')
X_test_sample = X_test.sample(n=10000, random_state=42)
y_test_sample = y_test.loc[X_test_sample.index]

# 텍스트 피처가 있으므로 Pool 형태로 묶어서 에러 방지 (필요 시)
sample_pool = Pool(
    X_test_sample, y_test_sample, cat_features=cat_features, text_features=text_features
)

result = permutation_importance(
    final_model,
    X_test_sample,  # Pool이 아닌 DataFrame 자체를 넣어야 사이킷런과 호환됨
    y_test_sample,
    n_repeats=6,
    random_state=42,
    scoring='roc_auc',
    n_jobs=-1,
)

perm_imp_df = pd.DataFrame(
    {
        'Feature': X_test_sample.columns,
        'Importance': result.importances_mean,
        'Std': result.importances_std,
    }
).sort_values(by='Importance', ascending=False)

print('\n[Permutation Importance 결과]')
print(perm_imp_df)

# 최종 모델 저장 (선택)
final_model.save_model("daangn_sell_predictor_new.cbm")

[I 2026-03-10 12:40:24,739] A new study created in memory with name: no-name-1878b5f0-ada5-43bd-b1f3-c759f17bd5f1


🚀 Optuna 하이퍼파라미터 탐색을 시작합니다...


[I 2026-03-10 12:40:40,954] Trial 0 finished with value: 0.9960713608509496 and parameters: {'iterations': 961, 'learning_rate': 0.023533088529156754, 'depth': 4, 'l2_leaf_reg': 0.04182968205336778, 'random_strength': 0.05864638400883813, 'bagging_temperature': 0.9577969640532855}. Best is trial 0 with value: 0.9960713608509496.
[I 2026-03-10 12:40:54,624] Trial 1 finished with value: 0.9960793916728806 and parameters: {'iterations': 703, 'learning_rate': 0.05658673349811275, 'depth': 6, 'l2_leaf_reg': 0.08927360664079807, 'random_strength': 5.164391608548683, 'bagging_temperature': 0.8544076809692893}. Best is trial 1 with value: 0.9960793916728806.
[I 2026-03-10 12:41:06,411] Trial 2 finished with value: 0.995824253714907 and parameters: {'iterations': 1233, 'learning_rate': 0.04688856057068168, 'depth': 7, 'l2_leaf_reg': 0.03813690873871569, 'random_strength': 0.08192320702359589, 'bagging_temperature': 0.9301864000054779}. Best is trial 1 with value: 0.9960793916728806.
[I 2026-03-

In [ ]:
# 1. 트리 모델용 초고속 SHAP Explainer 선언
explainer = shap.TreeExplainer(final_model)

# 2. Test Set 일부에 대한 SHAP 값 계산 (속도가 빨라 더 많이 넣어도 됩니다)
X_test_sample = X_test.sample(n=10000, random_state=42)
# shap_values = explainer(X_test_sample)
shap_values = explainer.shap_values(sample_pool)

# 3. 전체 Feature Importance 시각화 (요약 플롯)
# 피처가 예측값(판매 확률)을 높이는 데 기여했는지 낮추는 데 기여했는지 색상으로 보여줍니다.
shap.summary_plot(shap_values, X_test_sample)

# 4. 단순 막대 그래프로 중요도 순위만 보고 싶을 때
shap.summary_plot(shap_values, X_test_sample, plot_type='bar')

In [ ]:
# (한글 폰트 깨짐 방지 설정 - 윈도우 맑은고딕 기준)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 1. SHAP TreeExplainer 선언
explainer = shap.TreeExplainer(final_model)

# 2. 분석하고 싶은 데이터 딱 1개 고르기
best_idx = np.argmax(final_model.predict_proba(X_test)[:, 1])
sample_data = X_test.iloc[[best_idx]]
sample_target = y_test.iloc[best_idx]
sample_proba = final_model.predict_proba(sample_data)[0, 1]

# 3. 텍스트 피처 에러 방지를 위한 Pool 객체로 감싸기
sample_pool = catboost.Pool(
    sample_data, cat_features=cat_features, text_features=text_features
)

# 4. SHAP 값 계산
shap_values = explainer(sample_pool)

# ★ 에러 해결의 핵심: SHAP 객체 내부의 'Pool' 데이터를 실제 텍스트/숫자 데이터로 교체 ★
shap_values.data = sample_data.values
shap_values.feature_names = sample_data.columns.tolist()

# 5. 결과 출력 및 Waterfall 차트 시각화
print('==========================================')
print(f'판매 아이템: {sample_data["title"].values[0]}')
print(f'본문 내용: {sample_data["content"].values[0][:100]}...')
print(f'실제 7일 내 판매 여부: {"✅ 팔림" if sample_target == 1 else "❌ 안 팔림"}')
print(f'🔥 AI 예측 판매 확률: {sample_proba * 100:.2f}%')
print('==========================================\n')

# 폭포수(Waterfall) 차트 그리기
shap.plots.waterfall(shap_values[0], max_display=10)  # 상위 10개 피처만 깔끔하게 보기

In [ ]:
# ---------------------------------------------------------
# [1] 모델 저장 및 필수 매핑(Mapping) 데이터 준비
# ---------------------------------------------------------
# 1. 학습된 CatBoost 모델 저장
final_model.save_model('../data/models/daangn_sell_predictor.cbm')

In [ ]:
# 2. '가성비' 계산을 위한 평균 가격 정보 딕셔너리화 (매우 중요!)
# 실제 서비스에서는 이 딕셔너리들을 pickle 파일이나 DB에 저장해두고 불러와서 씁니다.
#brand_mean_dict = train_df.groupby('brandName')['price'].mean().to_dict()
#label_mean_dict = train_df.groupby('label')['price'].mean().to_dict()


# ---------------------------------------------------------
# [2] 새로운 게시글 예측 함수 만들기
# ---------------------------------------------------------
def predict_sell_probability(
    title,
    content,
    price,
    brandName,
    label,
    region_name,
    created_at_str,
    seller_temp=36.5,
):
    """
    새로운 게시글 정보를 입력받아 n일 이내 판매 확률을 반환하는 함수
    """
    # 1. 모델 불러오기 (실제 환경에서는 서버가 켜질 때 한 번만 불러옵니다)
    loaded_model = CatBoostClassifier()
    loaded_model.load_model('../data/models/daangn_sell_predictor.cbm')

    # 2. 파생 변수(가성비) 계산
    # 딕셔너리에 없는 새로운 브랜드/라벨이 들어오면 내 가격을 평균으로 간주 (비율이 1이 되도록)
    b_mean = brand_mean_dict.get(brandName, price)
    l_mean = label_mean_dict.get(label, price)

    price_ratio_to_brand = price / (b_mean + 1)
    price_ratio_to_label = price / (l_mean + 1)

    title_len = len(str(title))
    has_keyword_new = 1 if re.search(r'새상품|미개봉|새제품|택포', str(title)) else 0

    # 3. 모델에 넣을 데이터프레임 구성 (학습 때 사용한 features 리스트와 정확히 동일해야 함!)
    # --- 모델 입력용 데이터프레임 조립 ---
    input_data = pd.DataFrame(
        [
            {
                'price': price,
                'sellerTemperature': seller_temp,
                'price_ratio_to_brand': price_ratio_to_brand,
                'price_ratio_to_label': price_ratio_to_label,
                'title_len': title_len,
                'has_keyword_new': has_keyword_new,
                'title': title,
                'content': content,
                'region_name': region_name if pd.notnull(region_name) else 'unknown',
            }
        ]
    )

    # 결측치 처리 (학습 때와 동일하게)
    input_data['region_name'] = input_data['region_name'].fillna('unknown')

    # 4. 예측 수행
    proba = loaded_model.predict_proba(input_data)[0, 1]

    print('==========================================')
    print(f'판매 아이템: {title}')
    print(f'희망 가격: {price:,}원 (브랜드 평균 대비 {price_ratio_to_brand:.2f}배)')
    print(f'🔥 AI 예측 판매 확률: {proba * 100:.2f}%')
    print('==========================================\n')

    return proba


# ---------------------------------------------------------
# [3] 가상 데이터로 즉석 테스트 해보기
# ---------------------------------------------------------
# 가상 데이터 1: 매력적인 조건의 글
predict_sell_probability(
    title='나이키 숏패딩 블랙 M 급처합니다',
    content='선물 받았는데 사이즈가 안 맞아서 팝니다. 택 안 뗀 새상품이에요. 쿨거래 시 택포해드림.',
    price=35000,
    brandName='nike',
    label='padding jacket',
    region_name='역삼동',
    seller_temp=40.2,
    created_at_str='2024-06-01',
)

# 가상 데이터 2: 안 팔릴 것 같은 글
predict_sell_probability(
    title='나이키 패딩',
    content='입을만 합니다 네고 불가',
    price=150000,
    brandName='nike',
    label='padding jacket',
    region_name='역삼동',
    seller_temp=36.5,
    created_at_str='2024-06-01',
)

In [ ]:
# 1. 전체 Test 데이터에 예측 확률과 실제 정답 붙이기
analysis_df = X_test.copy()
analysis_df['actual_target'] = y_test
analysis_df['pred_proba'] = final_model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# [유형 1] False Positive (FP): 모델은 무조건 팔린다고(85% 이상) 했는데, 안 팔린 상품
# ---------------------------------------------------------
false_positives = analysis_df[
    (analysis_df['actual_target'] == 0) & (analysis_df['pred_proba'] >= 0.85)
].sort_values(by='pred_proba', ascending=False)

print(
    f'🚨 [False Positives] 모델은 팔린다고 확신했지만 안 팔린 게시글: {len(false_positives)}개'
)

# 상위 5개 뽑아보기
for idx, row in false_positives.head(5).iterrows():
    # ★ 수정된 부분: X_test에 없는 브랜드명은 원본 train_df에서 인덱스(idx)로 가져옵니다.
    original_brand = clean_df.loc[idx, 'brandName']

    print(
        f'확률: {row["pred_proba"]:.2f} | 가격: {row["price"]:,.0f}원 | 브랜드: {original_brand}'
    )
    print(f'제목: {row["title"]}')
    print(f'본문: {row["content"][:100]}...')  # 본문은 100자까지만
    print('-' * 50)


print('\n\n')


# ---------------------------------------------------------
# [유형 2] False Negative (FN): 모델은 절대 안 팔린다고(10% 이하) 했는데, 팔린 상품
# ---------------------------------------------------------
false_negatives = analysis_df[
    (analysis_df['actual_target'] == 1) & (analysis_df['pred_proba'] <= 0.10)
].sort_values(by='pred_proba', ascending=True)

print(
    f'🚨 [False Negatives] 모델은 안 팔린다고 확신했지만 실제론 팔린 게시글: {len(false_negatives)}개'
)

# 하위 5개 뽑아보기
for idx, row in false_negatives.head(5).iterrows():
    # ★ 수정된 부분
    original_brand = clean_df.loc[idx, 'brandName']

    print(
        f'확률: {row["pred_proba"]:.2f} | 가격: {row["price"]:,.0f}원 | 브랜드: {original_brand}'
    )
    print(f'제목: {row["title"]}')
    print(f'본문: {row["content"][:100]}...')
    print('-' * 50)